In [1]:
import pandas as pd
import geopandas as gpd

# Load cleaned crime data
crime = pd.read_csv('../data/processed/crime_clean.csv')

# Load neighborhood boundaries
neighborhoods = gpd.read_file('../data/raw/neighborhoods.geojson')

print(crime.shape)
print(neighborhoods.shape)
neighborhoods.head()

(236447, 6)
(26, 9)


,OBJECTID,name,acres,neighborhood_id,sqmiles,Shape_Length,Shape_Area,shape_wkt,geometry
0,1,Roslindale,1605.568237,15,2.51,0.169120,0.000709,None,"MULTIPOLYGON (((-71.12593 42.272, -71.12611 42..."
1,2,Jamaica Plain,2519.245394,11,3.94,0.179578,0.001113,None,"POLYGON ((-71.10499 42.32609, -71.10503 42.326..."
2,3,Mission Hill,350.853564,13,0.55,0.059235,0.000155,None,"POLYGON ((-71.09043 42.33576, -71.0905 42.3357..."
3,4,Longwood,188.611947,28,0.29,0.038816,0.000083,None,"POLYGON ((-71.09811 42.33672, -71.09832 42.337..."
4,5,Bay Village,26.539839,33,0.04,0.015625,0.000012,None,"POLYGON ((-71.06663 42.34877, -71.06663 42.348..."


In [9]:
crime_geo = gpd.GeoDataFrame( crime, geometry=gpd.points_from_xy(crime['Long'], crime['Lat']))

In [10]:
print(type(crime_geo))
print(crime_geo.shape)
crime_geo.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
(236447, 7)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166)
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126)
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657)
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279)
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027)


In [11]:
print(crime_geo.crs)
print(neighborhoods.crs)

None
EPSG:4326


In [12]:
crime_geo = crime_geo.set_crs(epsg=4326)
print(crime_geo.crs)

EPSG:4326


In [13]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo, 
    neighborhoods[['name', 'geometry']], 
    how='left', 
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods.head()

(236448, 9)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry,index_right,name
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166),20.0,Mattapan
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126),8.0,Roxbury
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657),7.0,North End
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279),21.0,Dorchester
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027),8.0,Roxbury


In [14]:
print(neighborhoods.total_bounds)
print(crime_geo.total_bounds)

[-71.1912491   42.22791113 -70.92277988  42.3969775 ]
[-71.22812796  42.17145601 -70.83688105  42.45998696]


In [15]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo,
    neighborhoods[['name', 'geometry']],
    how='left',
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods['name'].value_counts()



(236448, 9)


name
Dorchester                 54640
Roxbury                    28153
South End                  15430
South Boston               14064
Downtown                   13603
Jamaica Plain              13448
East Boston                12496
Brighton                   11415
Hyde Park                  10070
Back Bay                    9813
Mattapan                    8193
West Roxbury                7823
Fenway                      5658
Allston                     5407
Roslindale                  4955
West End                    4103
Charlestown                 3948
Mission Hill                3348
Beacon Hill                 2604
North End                   1971
South Boston Waterfront     1904
Chinatown                   1513
Longwood                    1117
Bay Village                  356
Leather District             313
Harbor Islands                11
Name: count, dtype: int64

In [16]:
crime_neighborhoods = crime_with_neighborhoods[['OFFENSE_DESCRIPTION', 'SHOOTING', 'OCCURRED_ON_DATE', 'Lat', 'Long', 'name']].copy()
crime_neighborhoods = crime_neighborhoods.rename(columns={'name': 'neighborhood'})
crime_neighborhoods.head()

,OFFENSE_DESCRIPTION,SHOOTING,OCCURRED_ON_DATE,Lat,Long,neighborhood
0,INVESTIGATE PERSON,0,2023-01-27 22:44:00+00,42.271661,-71.099534,Mattapan
1,VERBAL DISPUTE,0,2023-01-17 20:21:00+00,42.312597,-71.092875,Roxbury
2,INVESTIGATE PERSON,0,2023-01-24 00:00:00+00,42.365700,-71.052892,North End
3,INVESTIGATE PROPERTY,0,2023-03-31 17:14:00+00,42.292788,-71.088519,Dorchester
4,ASSAULT - AGGRAVATED,0,2023-01-26 09:00:00+00,42.310269,-71.089310,Roxbury


In [17]:
crime_neighborhoods.to_csv('../data/processed/crime_with_neighborhoods.csv', index=False)

In [18]:
mbta = gpd.read_file('../data/raw/MBTA_NODE.shp')
print(mbta.shape)
print(mbta.columns.tolist())
mbta.head()

(170, 5)
['STATION', 'LINE', 'TERMINUS', 'ROUTE', 'geometry']


,STATION,LINE,TERMINUS,ROUTE,geometry
0,Park Street,GREEN/RED,N,GREEN B C D E / RED A - Ashmont B - Braintree...,POINT (236064.005 900737.761)
1,JFK/UMass,RED,N,A - Ashmont B - Braintree C - Alewife,POINT (236899.089 896780.048)
2,State,BLUE/ORANGE,N,BLUE Bowdoin to Wonderland / ORANGE Forest Hil...,POINT (236427.189 901016.63)
3,Roxbury Crossing,ORANGE,N,Forest Hills to Oak Grove,POINT (233318.703 897936.59)
4,Airport Terminal B2,SILVER,N,SL1,POINT (239674.505 901509.172)


In [19]:
print(mbta.crs)

EPSG:26986


In [20]:
mbta = mbta.to_crs(epsg=4326)
print(mbta.crs)
mbta.head()

EPSG:4326


,STATION,LINE,TERMINUS,ROUTE,geometry
0,Park Street,GREEN/RED,N,GREEN B C D E / RED A - Ashmont B - Braintree...,POINT (-71.06224 42.35631)
1,JFK/UMass,RED,N,A - Ashmont B - Braintree C - Alewife,POINT (-71.05236 42.32064)
2,State,BLUE/ORANGE,N,BLUE Bowdoin to Wonderland / ORANGE Forest Hil...,POINT (-71.05782 42.3588)
3,Roxbury Crossing,ORANGE,N,Forest Hills to Oak Grove,POINT (-71.09573 42.33121)
4,Airport Terminal B2,SILVER,N,SL1,POINT (-71.01837 42.36308)


In [21]:
neighborhoods_proj = neighborhoods.to_crs(epsg=26986)
mbta_proj = mbta.to_crs(epsg=26986)

print(neighborhoods_proj.crs)
print(mbta_proj.crs)

EPSG:26986
EPSG:26986


In [22]:
neighborhoods_proj['centroid'] = neighborhoods_proj.geometry.centroid
print(neighborhoods_proj[['name', 'centroid']].head())

            name                       centroid
0     Roslindale  POINT (230792.997 892516.277)
1  Jamaica Plain  POINT (231734.222 895324.512)
2   Mission Hill  POINT (232751.627 897992.054)
3       Longwood  POINT (232542.397 898753.889)
4    Bay Village  POINT (235508.961 899933.943)


In [23]:
def count_nearby_stops(centroid, stops, radius=1609):
    return sum(stops.distance(centroid) <= radius)

neighborhoods_proj['transit_score'] = neighborhoods_proj['centroid'].apply(
    lambda c: count_nearby_stops(c, mbta_proj.geometry)
)

neighborhoods_proj[['name', 'transit_score']].sort_values('transit_score', ascending=False)

,name,transit_score
4,Bay Village,37
6,Chinatown,33
9,South End,32
5,Leather District,30
14,Beacon Hill,29
10,Back Bay,29
15,Downtown,29
3,Longwood,24
16,Fenway,21
13,West End,19


In [24]:
def count_nearby_stops_boundary(neighborhood_geom, stops, radius=1609):
    return sum(stops.distance(neighborhood_geom) <= radius)

neighborhoods_proj['transit_score'] = neighborhoods_proj.geometry.apply(
    lambda geom: count_nearby_stops_boundary(geom, mbta_proj.geometry)
)

neighborhoods_proj[['name', 'transit_score']].sort_values('transit_score', ascending=False)

,name,transit_score
10,Back Bay,57
23,South Boston,55
16,Fenway,53
9,South End,51
15,Downtown,47
8,Roxbury,45
22,South Boston Waterfront,44
2,Mission Hill,43
6,Chinatown,43
4,Bay Village,41


In [25]:
from shapely.geometry import Point

# BU campus center point
bu_point = gpd.GeoDataFrame(
    geometry=[Point(-71.1054, 42.3505)], 
    crs='EPSG:4326'
).to_crs(epsg=26986).geometry[0]

# Distance from each neighborhood centroid to BU in miles
neighborhoods_proj['bu_distance_miles'] = neighborhoods_proj['centroid'].apply(
    lambda c: c.distance(bu_point) / 1609
).round(2)

neighborhoods_proj[['name', 'transit_score', 'bu_distance_miles']].sort_values('bu_distance_miles')

,name,transit_score,bu_distance_miles
16,Fenway,53,0.62
3,Longwood,38,0.82
10,Back Bay,57,1.27
2,Mission Hill,43,1.30
24,Allston,35,1.31
9,South End,51,1.85
4,Bay Village,41,1.86
14,Beacon Hill,41,1.95
6,Chinatown,43,2.24
13,West End,29,2.26


In [26]:
transit_bu_scores = neighborhoods_proj[['name', 'transit_score', 'bu_distance_miles']].copy()
transit_bu_scores.to_csv('../data/processed/transit_bu_scores.csv', index=False)
print("Saved!")

Saved!


In [27]:
rent = pd.read_csv('../data/raw/ZORI_AllHomesPlusMultifamily_SSA.csv')
print(rent.shape)
print(rent.columns.tolist()[:10])
rent.head()

(7782, 142)
['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName', '2015-01-31']


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2015-01-31,...,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,1451.089344,...,1839.947583,1842.985657,1842.381563,1849.986968,1855.353309,1849.102029,1839.240212,1826.238505,1822.140686,1803.813488
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2385.000000
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,1240.398196,...,1846.108218,1840.913132,1832.919774,1828.631792,1819.567524,1810.683358,1809.883944,1799.512657,1810.114483,1800.547424
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2185.628181,2304.750000
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,1116.808168,...,1625.805489,1607.863981,1606.915864,1602.443131,1609.637147,1608.350155,1602.471577,1590.745289,1584.603251,1585.098746


In [28]:
rent[rent['State'] == 'MA']['City'].value_counts().head(20)

City
Boston         25
Worcester       8
Cambridge       5
Lowell          4
Springfield     4
Quincy          3
Lynn            3
Brookline       3
Waltham         3
Fall River      3
Somerville      3
Haverhill       3
Weymouth        3
Lawrence        2
New Bedford     2
Framingham      2
Billerica       2
Arlington       2
Chelmsford      2
Newton          2
Name: count, dtype: int64

In [29]:
boston_rent = rent[(rent['State'] == 'MA') & (rent['City'] == 'Boston')].copy()
print(boston_rent.shape)
boston_rent[['RegionName', 'City', 'CountyName']].head(25)

(25, 142)


,RegionName,City,CountyName
564,2124,Boston,Suffolk County
1246,2128,Boston,Suffolk County
1383,2135,Boston,Suffolk County
1776,2130,Boston,Suffolk County
1913,2136,Boston,Suffolk County
2529,2127,Boston,Suffolk County
2930,2125,Boston,Suffolk County
3026,2131,Boston,Suffolk County
3560,2121,Boston,Suffolk County
3671,2115,Boston,Suffolk County


In [30]:
print(rent.columns[-5:].tolist())

['2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31']


In [31]:
boston_rent = boston_rent[['RegionName', '2026-01-31']].copy()
boston_rent.columns = ['zipcode', 'avg_rent']
boston_rent['zipcode'] = boston_rent['zipcode'].astype(str)
print(boston_rent)

     zipcode     avg_rent
564     2124  3012.981481
1246    2128  3042.503251
1383    2135  3031.608842
1776    2130  3120.935185
1913    2136  2674.583333
2529    2127  3511.011438
2930    2125  3184.948580
3026    2131  2817.138889
3560    2121  3080.555556
3671    2115  4062.236259
3701    2119  3734.493687
3762    2118  4024.547438
4172    2132  2800.422222
4501    2215  3465.012575
4768    2122  2914.285714
5117    2116  3446.915761
5205    2134  3653.712832
5486    2129  3326.277778
6592    2120  5617.628520
6812    2114  3627.857168
7443    2111  3749.526282
7551    2113  3156.325052
7647    2210  4093.264444
7656    2109  3826.670635
7665    2108  3433.750000


In [32]:
boston_rent['zipcode'] = boston_rent['zipcode'].str.zfill(5)
print(boston_rent['zipcode'].tolist())

['02124', '02128', '02135', '02130', '02136', '02127', '02125', '02131', '02121', '02115', '02119', '02118', '02132', '02215', '02122', '02116', '02134', '02129', '02120', '02114', '02111', '02113', '02210', '02109', '02108']


In [33]:
zip_to_neighborhood = {
    '02108': 'Beacon Hill',
    '02109': 'North End',
    '02110': 'Downtown',
    '02111': 'Chinatown',
    '02113': 'North End',
    '02114': 'West End',
    '02115': 'Fenway',
    '02116': 'Back Bay',
    '02118': 'South End',
    '02119': 'Roxbury',
    '02120': 'Mission Hill',
    '02121': 'Dorchester',
    '02122': 'Dorchester',
    '02124': 'Dorchester',
    '02125': 'Dorchester',
    '02126': 'Mattapan',
    '02127': 'South Boston',
    '02128': 'East Boston',
    '02129': 'Charlestown',
    '02130': 'Jamaica Plain',
    '02131': 'Roslindale',
    '02132': 'West Roxbury',
    '02134': 'Allston',
    '02135': 'Brighton',
    '02136': 'Hyde Park',
    '02210': 'South Boston Waterfront',
    '02215': 'Fenway'
}

boston_rent['neighborhood'] = boston_rent['zipcode'].map(zip_to_neighborhood)
print(boston_rent)

     zipcode     avg_rent             neighborhood
564    02124  3012.981481               Dorchester
1246   02128  3042.503251              East Boston
1383   02135  3031.608842                 Brighton
1776   02130  3120.935185            Jamaica Plain
1913   02136  2674.583333                Hyde Park
2529   02127  3511.011438             South Boston
2930   02125  3184.948580               Dorchester
3026   02131  2817.138889               Roslindale
3560   02121  3080.555556               Dorchester
3671   02115  4062.236259                   Fenway
3701   02119  3734.493687                  Roxbury
3762   02118  4024.547438                South End
4172   02132  2800.422222             West Roxbury
4501   02215  3465.012575                   Fenway
4768   02122  2914.285714               Dorchester
5117   02116  3446.915761                 Back Bay
5205   02134  3653.712832                  Allston
5486   02129  3326.277778              Charlestown
6592   02120  5617.628520      

In [34]:
neighborhood_rent = boston_rent.groupby('neighborhood')['avg_rent'].mean().round(2).reset_index()
print(neighborhood_rent.sort_values('avg_rent'))

               neighborhood  avg_rent
9                 Hyde Park   2674.58
19             West Roxbury   2800.42
13               Roslindale   2817.14
3                  Brighton   3031.61
7               East Boston   3042.50
6                Dorchester   3048.19
10            Jamaica Plain   3120.94
4               Charlestown   3326.28
2               Beacon Hill   3433.75
1                  Back Bay   3446.92
12                North End   3491.50
15             South Boston   3511.01
18                 West End   3627.86
0                   Allston   3653.71
14                  Roxbury   3734.49
5                 Chinatown   3749.53
8                    Fenway   3763.62
17                South End   4024.55
16  South Boston Waterfront   4093.26
11             Mission Hill   5617.63


In [35]:
boston_rent[boston_rent['neighborhood'] == 'Mission Hill']

,zipcode,avg_rent,neighborhood
6592,02120,5617.62852,Mission Hill


In [36]:
print(neighborhood_rent['avg_rent'].describe())

count      20.000000
mean     3500.474500
std       640.283846
min      2674.580000
25%      3046.767500
50%      3469.210000
75%      3738.250000
max      5617.630000
Name: avg_rent, dtype: float64


In [37]:
print(f"Neighborhoods with rent data: {len(neighborhood_rent)}")
print(f"Total neighborhoods: {len(neighborhoods)}")

# Which neighborhoods are MISSING rent data?
all_neighborhoods = set(neighborhoods['name'].tolist())
rent_neighborhoods = set(neighborhood_rent['neighborhood'].tolist())
missing = all_neighborhoods - rent_neighborhoods
print(f"Missing: {missing}")

Neighborhoods with rent data: 20
Total neighborhoods: 26
Missing: {'Mattapan', 'Bay Village', 'Leather District', 'Downtown', 'Longwood', 'Harbor Islands'}


In [38]:
rent[rent['RegionName'] == 2126][['RegionName', 'City', 'StateName', '2026-01-31']]

,RegionName,City,StateName,2026-01-31


In [39]:
import osmnx as ox

# Query grocery stores in Boston
groceries = ox.features_from_place(
    "Boston, Massachusetts, USA",
    tags={'shop': 'supermarket'}
)

print(groceries.shape)
print(groceries.columns.tolist())

(60, 67)
['geometry', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:state', 'addr:street', 'brand', 'brand:wikidata', 'check_date', 'check_date:opening_hours', 'name', 'note', 'opening_hours', 'phone', 'ref', 'shop', 'website', 'addr:country', 'air_conditioning', 'currency:USD', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'operator', 'organic', 'outdoor_seating', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'stroller', 'takeaway', 'wheelchair', 'branch', 'level', 'toilets:wheelchair', 'designation', 'contact:facebook', 'payment:discover_card', 'payment:mastercard', 'payment:visa', 'short_name', 'alt_name', 'name:en', 'addr:place', 'addr:unit', 'name:fr', 'building:levels:underground', 'layer', 'access', 'location', 'source', 'building', 'building:levels', 'payment:apple_pay', 'payment:cards', 'payment:contactless', 'addr:housename', 'height', 'old_name', 'building:material', 'roof:shape', 'contact:phone', 'operator:type', 'payment:ebt',

In [40]:
# Query multiple amenity types
groceries = ox.features_from_place("Boston, Massachusetts, USA", tags={'shop': 'supermarket'})
restaurants = ox.features_from_place("Boston, Massachusetts, USA", tags={'amenity': 'restaurant'})
cafes = ox.features_from_place("Boston, Massachusetts, USA", tags={'amenity': 'cafe'})

print(f"Grocery stores: {len(groceries)}")
print(f"Restaurants: {len(restaurants)}")
print(f"Cafes: {len(cafes)}")

Grocery stores: 60
Restaurants: 857
Cafes: 282


In [41]:
# Keep only name and geometry, add type label
groceries_clean = groceries[['name', 'geometry']].copy()
groceries_clean['type'] = 'grocery'

restaurants_clean = restaurants[['name', 'geometry']].copy()
restaurants_clean['type'] = 'restaurant'

cafes_clean = cafes[['name', 'geometry']].copy()
cafes_clean['type'] = 'cafe'

# Combine all
amenities = pd.concat([groceries_clean, restaurants_clean, cafes_clean])
print(amenities.shape)
print(amenities['type'].value_counts())


(1199, 3)
type
restaurant    857
cafe          282
grocery        60
Name: count, dtype: int64


In [42]:
print(amenities.crs)
print(neighborhoods.crs)

epsg:4326
EPSG:4326


In [43]:
amenities_with_neighborhoods = gpd.sjoin(
    amenities,
    neighborhoods[['name', 'geometry']],
    how='left',
    predicate='within'
)

amenity_counts = amenities_with_neighborhoods.groupby('name_right')['type'].count().reset_index()
amenity_counts.columns = ['neighborhood', 'amenity_count']
amenity_counts.sort_values('amenity_count', ascending=False)

,neighborhood,amenity_count
8,Downtown,165
1,Back Bay,125
0,Allston,121
9,East Boston,117
10,Fenway,99
4,Brighton,74
18,North End,64
22,South Boston Waterfront,56
23,South End,56
13,Jamaica Plain,49


In [44]:
amenity_counts.to_csv('../data/processed/amenity_scores.csv', index=False)
print("Saved!")

Saved!


In [45]:
neighborhood_rent.to_csv('../data/processed/rent_scores.csv', index=False)
print("Saved!")

Saved!
